# Интерпретация

In [421]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso,LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'


In [422]:
data_path = "C:\prog\ml\period_1_train_data.csv"
data = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {data.shape}")
print(f"\nПервые строки:")
data.head()


Размер датасета: (29905, 83)

Первые строки:


,region_name_cat,district_cat,corpus_cat,developer_cat,agreement_date,floor,square,rooms_4,location_logs_count_mean,location_depth,...,location_public_transport_platform_w_mean_distance,location_water_w_mean_distance,location_university_w_mean_distance,location_leisure_w_mean_distance,location_pop_shop_cnt,price_target,hc_name_cat,interior_cat,class_cat,stage_cat
0,Город,45,538,18,2012-08-10,3.0,62.23,2,22.550466,13.0,...,0.910028,0.782675,-999.000000,0.820073,16.0,28417.424671,50,49786.0,27353,7983
1,Пригород,48,432,63,2013-05-19,11.0,22.52,студия,22.581858,13.0,...,0.902510,0.902673,-999.000000,0.990908,18.0,16728.215463,293,49786.0,97865,70661
2,Город,44,2372,126,2012-12-12,3.0,38.17,1,20.191250,13.0,...,0.851637,-999.000000,-999.000000,0.945618,7.0,18311.834458,284,49786.0,97865,70661
3,Город,14,1053,121,2012-12-10,10.0,57.48,2,23.286900,13.0,...,0.913797,1.028386,0.300026,0.828147,5.0,25171.489968,325,0.0,97865,12638
4,Город,63,2426,69,2012-02-12,3.0,41.43,1,20.599150,13.0,...,1.051049,-999.000000,-999.000000,0.991506,4.0,27324.795343,182,49786.0,97865,70661


In [425]:
data_cop=data.copy()
data_cop=data_cop.dropna()

In [427]:
cat_feat=["region_name_cat","agreement_date","rooms_4"]
data_cop[cat_feat]
print(data_cop["region_name_cat"].value_counts())
print(data_cop["rooms_4"].value_counts())


region_name_cat
Город       22929
Пригород     6643
Name: count, dtype: int64
rooms_4
1         14025
2         10207
3          4378
>=4         631
студия      331
Name: count, dtype: int64


In [428]:
# Предобработка
region_map = {
    "Город": 2,
    "Пригород": 1,
    "Область": 0
}

rooms_map = {
    "студия": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    ">=4": 4
}


remade_dss = []
for dataset in [data_cop]:
    num_only_data = dataset
    
    # Преобразование категориальных признаков
    num_only_data['region_name_cat'] = num_only_data['region_name_cat'].map(region_map)
    num_only_data['rooms_4'] = num_only_data['rooms_4'].map(rooms_map)
    
    # Извлечение даты (более надежный способ)
    num_only_data['agreement_date'] = pd.to_datetime(num_only_data['agreement_date'])
    num_only_data['Year'] = num_only_data['agreement_date'].dt.year.astype(np.int64)
    num_only_data['Month'] = num_only_data['agreement_date'].dt.month.astype(np.int64)
    num_only_data['Day'] = num_only_data['agreement_date'].dt.day.astype(np.int64)
    
    # Переименование колонок
    num_only_data.rename(columns={'region_name_cat': 'region_name'}, inplace=True)
    
    # Удаление исходной колонки с датой
    num_only_data.drop('agreement_date', axis=1, inplace=True)
    
    remade_dss.append(num_only_data)

print(remade_dss[0][['Year', 'Month', 'Day']].head())

   Year  Month  Day
0  2012      8   10
1  2013      5   19
2  2012     12   12
3  2012     12   10
4  2012      2   12


In [429]:
X = data_cop.drop("price_target",axis=1).copy()
y=data_cop["price_target"].copy()

In [430]:
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import StackingRegressor
import xgboost as xgb
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,train_size=0.8, random_state=42)
lin_model = HistGradientBoostingRegressor(loss="absolute_error",max_depth=5,early_stopping=True,max_iter=100000,random_state=42,scoring='loss',learning_rate=0.05)
#lin_model.fit(X_train,y_train)
#y_pred = lin_model.predict(X_test)
#y_pred_train = lin_model.predict(X_train)


In [431]:
#X_train2 = X_train.astype(np.float32)
#X_test2 = X_test.astype(np.float32)
stack_mod = xgb.XGBRegressor(
    device='cuda',
    tree_method='hist',
    max_depth=7,
    learning_rate=0.03,
    n_estimators=1000,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=3,
    reg_alpha=1,
    n_jobs=-1,
    verbosity=1
)
     
#stack_mod.fit(X_train2,y_train)
#y_pred_stack_mod = stack_mod.predict(X_test2)
#_pred_stack_mod_train= stack_mod.predict(X_train2)

In [434]:
model_test=ExtraTreesRegressor(max_depth=50, n_estimators=300, n_jobs=-1)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
weights = np.where(y_train > np.percentile(y_train, 90), 1.05, 1.0)  
X_stup_test=X_test.copy()
X_stup_train=X_train.copy()
y_stup_train = y_train_log
model_test.fit(X_stup_train,y_stup_train,sample_weight=weights)
y_pred_test=model_test.predict(X_stup_test)  
y_pred_train=model_test.predict(X_stup_train) 

print("Mape обычного RF",mean_absolute_percentage_error(np.expm1(y_test_log), np.expm1(y_pred_test)))
print("Mape обычного RF на трейне",mean_absolute_percentage_error(np.expm1(y_stup_train), np.expm1(y_pred_train)))


Mape обычного RF 0.017648869036074346
Mape обычного RF на трейне 7.674914640598238e-11


In [ ]:
preds = []

for seed in [42, 52, 62, 72,23,3,4,2,1,7,8,9]:
    model = model_test
    model.fit(X_train, y_train_log)
    pred = np.expm1(model.predict(X_test))

    preds.append(pred)

final_pred = np.median(preds, axis=0)

In [ ]:
print("Mape обычного RF",mean_absolute_percentage_error(y_test, final_pred))

Mape обычного RF 0.017367532253256265


In [ ]:
print("Mape стека из них на тест",mean_absolute_percentage_error(y_test, y_pred_stac))
print("Mape стека из них на траин",mean_absolute_percentage_error(y_train, y_pred_train_stac))

Mape стека из них на тест 0.02055204235014482
Mape стека из них на траин 0.008427714248135784


In [ ]:
from sklearn.ensemble import VotingRegressor
voting_model = VotingRegressor(
    estimators=estimators,
)

voting_model.fit(X_train, y_train)
y_pred_stac_vote = voting_model.predict(X_test)
y_pred_train_stac_vote = voting_model.predict(X_train)

In [ ]:
print("Mape обычного RF",mean_absolute_percentage_error(y_test, y_pred_stack_mod))
print("Mape train RF",mean_absolute_percentage_error(y_train, y_pred_stack_mod_train))
print("Mape обычного гардиентного бустинга",mean_absolute_percentage_error(y_test, y_pred))
print("Mape на обучающей выборке",mean_absolute_percentage_error(y_train, y_pred_train))
print("Mape стека Голосования из них на тест",mean_absolute_percentage_error(y_test, y_pred_stac_vote))
print("Mape стека Голосования из них на траин",mean_absolute_percentage_error(y_train, y_pred_train_stac_vote))


Mape обычного RF 0.027331214338686523
Mape train RF 0.021302177713872463
Mape обычного гардиентного бустинга 0.02818503686291285
Mape на обучающей выборке 0.9995533530175861
Mape стека Голосования из них на тест 0.021571601864407153
Mape стека Голосования из них на траин 0.010651594099772378


In [379]:
data_path = "C:\prog\ml\\test_x.csv"
test_data = pd.read_csv(data_path, sep=',')
test_data['Year'] = test_data["agreement_date"].str.extract('(\d+).').astype(np.int64)
test_data['Month'] = test_data.agreement_date.str.extract('.....(\d+).').astype(np.int8)
test_data['Day'] = test_data.agreement_date.str.extract('........(\d+)').astype(np.int8)
test_data['region_name_cat'] = test_data['region_name_cat'].map(title_mapping)
test_data=test_data.drop("agreement_date",axis=1)
test_data2=test_data
test_data2=test_data2.drop("id",axis=1)
test_data2=test_data2.drop(features_to_drop,axis=1)
test_data2['rooms_4'] = test_data2['rooms_4'].replace(mapping).astype(int)
preds = []
weights = np.where(y > np.percentile(y, 90), 1.05, 1.0)  
for seed in [42, 52, 62, 72,23,3,4,2,1,7,8,9]:
    model = ExtraTreesRegressor(n_estimators=566,n_jobs=-1,max_depth=36,random_state=seed,min_samples_leaf=1,min_samples_split=2,criterion="squared_error")

    model.fit(X,np.log1p(y),sample_weight=weights)
    pred = np.expm1(model.predict(test_data2))
    preds.append(pred)

final_pred = np.mean(preds, axis=0)
# sub=model_test.predict(test_data2)
# sub=np.expm1(sub)
submission = pd.DataFrame({
    'id': test_data['id'],        
    'price_target': final_pred    
})
submission.to_csv('submission2.csv', index=False)